# Initial Situation

In an effort to compare user reviews to journalist reviews and identify potential discrepancies or instances of reviews made in bad faith, we are undertaking a data collection project. This involves gathering a comprehensive dataset of reviews from OpenCritic, focusing initially on reviews authored by journalists from IGN. By systematically retrieving and analyzing this data, we aim to highlight significant differences between professional and user-generated reviews, thereby uncovering any biases or irregularities that may exist within professional reviews. The ultimate goal is to ensure greater transparency and accountability in the review process, fostering a more trustworthy and reliable gaming review ecosystem.

## Dependancies

In [2]:
import json
import requests
import urllib.parse
import pandas as pd

#BeatifulSoup for Webscraping
from bs4 import BeautifulSoup

# Kafka & Spark
# from kafka import KafkaProducer
# from pyspark.sql import SparkSession
# from pyspark.sql.functions import *
# from pyspark.sql.types import *

## Web Scraping

In order to compare user review data and journalist data, we first need to gather all this data. We use three different means of gathering data:

- Gather already collected Data that was provided on the internet via CSV-files
  - Gather Data for 1000 Games
- Using user created APIs to collect data from a Website:
  - Gather Outlet Reviews for the 1000 chosen Games, use the API to get the ID for each Game
- Web Scraping directly using a method
  - Gather User Reviews for the 1000 chosen Games

## Gather 1000 Games

To start this project, we first need to gather data for 1000 Games which will be the subject of our analysis. To do this, we have taken an existing dataset from Kaggle (https://www.kaggle.com/datasets/arnabchaki/popular-video-games-1980-2023).

First, we will load the data of this set and check if it was successfull:

In [3]:
# Load the CSV file
data = pd.read_csv('data/games.csv')

# Display the first few rows of the dataframe to understand its structure
data.head()

,Unnamed: 0,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist
0,0,Elden Ring,"Feb 25, 2022","['Bandai Namco Entertainment', 'FromSoftware']",4.5,3.9K,3.9K,"['Adventure', 'RPG']","Elden Ring is a fantasy, action and open world...","[""The first playthrough of elden ring is one o...",17K,3.8K,4.6K,4.8K
1,1,Hades,"Dec 10, 2019",['Supergiant Games'],4.3,2.9K,2.9K,"['Adventure', 'Brawler', 'Indie', 'RPG']",A rogue-lite hack and slash dungeon crawler in...,['convinced this is a roguelike for people who...,21K,3.2K,6.3K,3.6K
2,2,The Legend of Zelda: Breath of the Wild,"Mar 03, 2017","['Nintendo', 'Nintendo EPD Production Group No...",4.4,4.3K,4.3K,"['Adventure', 'RPG']",The Legend of Zelda: Breath of the Wild is the...,['This game is the game (that is not CS:GO) th...,30K,2.5K,5K,2.6K
3,3,Undertale,"Sep 15, 2015","['tobyfox', '8-4']",4.2,3.5K,3.5K,"['Adventure', 'Indie', 'RPG', 'Turn Based Stra...","A small child falls into the Underground, wher...",['soundtrack is tied for #1 with nier automata...,28K,679,4.9K,1.8K
4,4,Hollow Knight,"Feb 24, 2017",['Team Cherry'],4.4,3K,3K,"['Adventure', 'Indie', 'Platform']",A 2D metroidvania with an emphasis on close co...,"[""this games worldbuilding is incredible, with...",21K,2.4K,8.3K,2.3K


Now let's check if there are any empty Datasets at points where we actually need data:

In [4]:
# Check for empty strings in the "Title" and "Team" columns
empty_names = data['Title'].isna().sum()
empty_publishers = data['Team'].isna().sum()

empty_names, empty_publishers

(0, 1)

Seems like there is only one missing value in "Team", let's remove that:

In [5]:
# Remove rows with empty "publisher" values
data_cleaned = data.dropna(subset=['Team'])

# Verify that the rows with empty "publisher" values have been removed
empty_publishers_after_cleanup = data_cleaned['Team'].isna().sum()
empty_publishers_after_cleanup

0

Now let's check for duplicates:

In [6]:
# Find duplicated titles
duplicated_titles = data_cleaned[data_cleaned.duplicated(subset=['Title'], keep=False)]

# Count the number of duplicates for each title
duplicate_counts = duplicated_titles['Title'].value_counts()

# Display the counts of duplicated titles
print("\nCounts of Duplicated Titles:")
print(duplicate_counts)


Counts of Duplicated Titles:
Title
Doom                               7
Dead Space                         5
Shadow of the Colossus             5
Resident Evil 2                    5
Demon's Souls                      4
                                  ..
Mass Effect Legendary Edition      2
Helltaker                          2
Pokémon Sword                      2
Crash Bandicoot N. Sane Trilogy    2
FIFA 13                            2
Name: count, Length: 279, dtype: int64


Seems like the dataset also has a few duplicates where we don't want any, lets remove them:

In [7]:
# Count the number of items in the 'Title' column for data_cleaned
total_titles_before = data_cleaned['Title'].count()

# Drop duplicates based on the 'Title' column
unique_games = data_cleaned.drop_duplicates(subset='Title')

# Count the number of items in the 'Title' column for unique_games
total_titles_after = unique_games['Title'].count()

# Print the counts before and after cleaning
print(f"Number of items in 'Title' column before cleaning: {total_titles_before}")
print(f"Number of items in 'Title' column after cleaning: {total_titles_after}")

Number of items in 'Title' column before cleaning: 1511
Number of items in 'Title' column after cleaning: 1098


Seems like the data is now good, we will save the Game Titles:

In [8]:
# Extract the first 1000 entries from the "name" column
game_names = unique_games['Title'].tolist()
game_names[:10] # Display the first 10 names to verify the extractin

['Elden Ring',
 'Hades',
 'The Legend of Zelda: Breath of the Wild',
 'Undertale',
 'Hollow Knight',
 'Minecraft',
 'Omori',
 'Metroid Dread',
 'Among Us',
 'NieR: Automata']

In [9]:
# Also extract the first 1000 entries for publishers
game_publisher = unique_games['Team'].tolist()
game_publisher[:10]

["['Bandai Namco Entertainment', 'FromSoftware']",
 "['Supergiant Games']",
 "['Nintendo', 'Nintendo EPD Production Group No. 3']",
 "['tobyfox', '8-4']",
 "['Team Cherry']",
 "['Mojang Studios']",
 "['OMOCAT', 'PLAYISM']",
 "['Nintendo', 'MercurySteam']",
 "['InnerSloth']",
 "['PlatinumGames', 'Square Enix']"]

## OpenCritic API

OpenCritic is a Website that hosts Reviews from journalists. With this API, we can filter for certain categories, most importantly via the author id. With this, we can collect a certain amount of review scores from different media outlets.
We will be collecting 1000 datasets for each of the following 10 media outlets, including their id used for the API filtering:

- IGN (56)
- PC Gamer (162)
- GamesRadar+ (91)
- Metro GameCentral (75)
- Eurogamer (114)
- Easy Allies (394)
- Game Informer (35)
- Polygon (87)
- GameSpot (32)
- Kotaku (276)

First, we need to gather the OpenCritic game ID for each game we want the review score of, to do that, we iterate through our games array and send each string to the API to search for the game ID:

In [27]:
# API key used for the API access
api_key = "64686b9a75msha576b74f8ff0b56p1a63ebjsnf20f69b4fac9"

# Function to get game IDs
def get_game_ids(game_names, api_key):
    all_game_ids = []
    filtered_game_names = []

    for game_name in game_names:
        encoded_game_name = urllib.parse.quote(game_name)
        url = f"https://opencritic-api.p.rapidapi.com/game/search?criteria={encoded_game_name}"

        headers = {
            "X-RapidAPI-Host": "opencritic-api.p.rapidapi.com",
            "X-RapidAPI-Key": api_key
        }

        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            data = response.json()
            if data:
                # Check if the name of the top search result matches the game name exactly
                top_game = data[0]
                if top_game['name'].lower() == game_name.lower():
                    top_game_id = top_game['id']
                    all_game_ids.append((game_name, top_game_id))
                    filtered_game_names.append(game_name)
                    print(f"Game ID for '{game_name}' is {top_game_id}")
                else:
                    print(f"No exact match found for '{game_name}'")
            else:
                print(f"No results found for '{game_name}'")
        else:
            print(f"Error searching for '{game_name}': ", response.status_code)

    return filtered_game_names, all_game_ids

# Call the method
filtered_game_names, game_ids = get_game_ids(game_names, api_key)

# Save game IDs to JSON file
data = {"filtered_game_names": filtered_game_names, "game_ids": game_ids}
with open("data/game_ids.json", "w") as json_file:
    json.dump(data, json_file)

print("Game IDs saved to data/game_ids.json")

Game ID for 'Elden Ring' is 12090
Game ID for 'Hades' is 10181
Game ID for 'The Legend of Zelda: Breath of the Wild' is 1548
Game ID for 'Undertale' is 1907
Game ID for 'Hollow Knight' is 4002
Game ID for 'Minecraft' is 194
Game ID for 'Omori' is 10743
Game ID for 'Metroid Dread' is 11621
Game ID for 'Among Us' is 10238
Game ID for 'NieR: Automata' is 3540
Game ID for 'Persona 5 Royal' is 8785
Game ID for 'Stray' is 13386
Game ID for 'God of War' is 5434
No exact match found for 'Portal 2'
Game ID for 'Bloodborne' is 76
Game ID for 'Celeste' is 5403
Game ID for 'Yakuza 0' is 3387
Game ID for 'Red Dead Redemption 2' is 3717
No exact match found for 'Portal'
Game ID for 'Super Mario Odyssey' is 4504
Game ID for 'Pokémon Legends: Arceus' is 12091
Game ID for 'Hi-Fi Rush' is 14227
Game ID for 'Metal Gear Rising: Revengeance' is 1014
Game ID for 'Grand Theft Auto V' is 163
Game ID for 'Cyberpunk 2077' is 8525
Game ID for 'God of War Ragnarök' is 12919
Game ID for 'Xenoblade Chronicles 3' is

We had to filter out some of the Game Titles, because not everything can be found by the OpenCritic API, let's look how many we have left since the beginning:

In [10]:
print(f"Amount of Games before filtering: {len(game_names)}")
print(f"Amount of Games after filtering: {len(filtered_game_names)}")

Amount of Games before filtering: 1098


NameError: name 'filtered_game_names' is not defined

Now for the next step, we need to gather the Gaming Platforms from the API:

In [20]:
url = "https://opencritic-api.p.rapidapi.com/platform"

headers = {
    "X-RapidAPI-Host": "opencritic-api.p.rapidapi.com",
    "X-RapidAPI-Key": "64686b9a75msha576b74f8ff0b56p1a63ebjsnf20f69b4fac9"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    platforms = response.json()
else:
    print(f"Error searching for Platforms: ", response.status_code)

with open("data/platforms.json", "w") as json_file:
    json.dump(data, json_file)

print("Gaming Platforms saved to data/platforms.json")

Gaming Platforms saved to data/platforms.json


We lost half of the Dataset, but this is still a good number to work with.

Now we are going to collect with the help of a Kafke producer all the needed Data for each of the outlet IDs as well as the average score from every critic from every game. Since the API only collects 20 Reviews at a Time, we will need a loop to go through 20 sets, then skip the 20 already found and start a new search, until we have gathered each review for each game:

In [1]:
!docker-compose up -d

#necessary libraries
!pip install kafka-python requests

# load important libraries
import json
import requests
from kafka import KafkaProducer
import time

 Container sara_-db-1  Running


In [ ]:
# API key used for the API access
api_key = "64686b9a75msha576b74f8ff0b56p1a63ebjsnf20f69b4fac9"

# List of outlet IDs
outlets = {
    "IGN": 56,
    "PC Gamer": 162,
    "GamesRadar+": 91,
    "Metro GameCentral": 75,
    "Eurogamer": 114,
    "Easy Allies": 394,
    "Game Informer": 35,
    "Forbes": 290,
    "GameSpot": 32,
    "Game Rant": 60
}

# Function to load platforms data from platforms.json
def load_platforms_data(filename="platforms.json"):
    with open(filename, 'r') as f:
        platforms_data = json.load(f)
    platforms_map = {platform['id']: platform['name'] for platform in platforms_data}
    return platforms_map

# Function to get reviews for each game ID
def fetch_game_reviews(game_name, game_id, api_key, sort_by="newest"):
    game_reviews = []
    skip = 0

    while True:
        url = f"https://opencritic-api.p.rapidapi.com/reviews/game/{game_id}?skip={skip}&sort={sort_by}"
        headers = {
            "X-RapidAPI-Host": "opencritic-api.p.rapidapi.com",
            "X-RapidAPI-Key": api_key
        }
        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            data = response.json()
            if not data:
                break  # No more reviews to fetch
            game_reviews.extend(data)
            skip += 20
        else:
            print(f"Error for game ID '{game_id}' at skip={skip}: ", response.status_code)
            break

    return [(game_name, review) for review in game_reviews]

# Create Kafka producer
producer = KafkaProducer(
    bootstrap_servers='localhost:29092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

TOPIC_NAME = 'game_reviews'
GAME_IDS = [
    ("Game1", 123),
    ("Game2", 456),
    # Add more games here as needed
]

try:
    while True:  # Continuously fetch data and send to Kafka topic
        for game_name, game_id in GAME_IDS:
            game_reviews = fetch_game_reviews(game_name, game_id, api_key)
            for review in game_reviews:
                producer.send(TOPIC_NAME, {'game_name': game_name, 'review': review})
                print(f"Data Sent: {game_name} - {review}")
        time.sleep(3600)  # Fetch new data every hour
except KeyboardInterrupt:
    print("Process interrupted")

Data Sent: Game1 - ('Game1', {'_id': '5d866e6893765d677873993f', 'magic': 0, 'isChosen': False, 'title': 'Dying Light Review: A Toothless Zombie Romp', 'publishedDate': '2018-10-17T00:00:00.000Z', 'externalUrl': 'https://gideonsgaming.com/2018/10/17/throw-back-review-dying-light/', 'snippet': "The free-running is fantastic, the combat feels excellent, and the crafting and skill system are well designed. Dying Light Plays well, and the story is entertaining, even if it is fairly basic. The gunplay definitely sours the experience when it comes up, and the lack of any real consequence for failure tarnishes every mechanic of the game. If that doesn't turn you off then there is a lot to love about this zombie romp", 'language': 'en-us', 'score': 78, 'npScore': 100, 'isQuoteManual': True, 'Authors': [{'_id': '64316a9934e32f8b008350bc', 'id': 5900, 'name': 'Joseph Pugh', 'image': True, 'imageSrc': {'og': 'critic/5900/o/542eqvCP.jpg', 'sm': 'critic/5900/gqcL3bBw.jpg', 'lg': 'critic/5900/k0OoRs

Now this Kafka consumer script continuously reads game review data from the game_reviews topic, processes the reviews to extract and aggregate relevant details, and last but not least saves the results to a JSON file. It ensures real-time data processing and aggregation, providing structured insights into game reviews by outlet and platform.

In [ ]:
import json
import datetime
from kafka import KafkaConsumer

# Function to process reviews and save to JSON
def process_and_save_reviews(reviews, outlets, platforms_map, path="data/opencritic_reviews.json"):
    result = {}

    # Dictionary to store platform scores and counts
    platform_scores = {}
    platform_counts = {}

    for game_name, review in reviews:
        # Add debug statement to check the structure of the review
        print(f"Processing review for game: {game_name}")
        print(f"Review structure: {review}")

        if not isinstance(review, dict):
            print(f"Skipping invalid review: {review}")
            continue

        if 'Outlet' not in review or not isinstance(review['Outlet'], dict):
            print(f"Skipping review with invalid Outlet data: {review}")
            continue

        outlet_id = review['Outlet']['id']
        outlet_name = review['Outlet']['name']
        score = review.get('score')
        created_at = review['createdAt'][:10]  # Extract the date part (YYYY-MM-DD)
        authors = review.get('Authors', [])
        author_name = authors[0]['name'] if authors else None

        # Extract platforms from the review
        platforms = review.get('Platforms', [])
        if platforms:
            for platform in platforms:
                platform_id = platform['id']
                if platform_id in platforms_map:
                    platform_name = platforms_map[platform_id]
                    if platform_name not in platform_scores:
                        platform_scores[platform_name] = []
                        platform_counts[platform_name] = 0
                    if isinstance(score, (int, float)):
                        platform_scores[platform_name].append(score)
                        platform_counts[platform_name] += 1

        if game_name not in result:
            result[game_name] = {"outlets": {}, "platforms": {}, "average_score": None}

        if outlet_id in outlets.values():
            # Check if the score is a valid number
            if isinstance(score, (int, float)) and score is not None:
                result[game_name]["outlets"][outlet_name] = {
                    "score": score,
                    "review_url": review['externalUrl'],
                    "created_at": created_at,
                    "author": author_name  # Include author's name if available
                }

    # Calculate average scores for each platform
    for game_name, data in result.items():
        for platform_name, scores in platform_scores.items():
            if scores:
                avg_score = sum(scores) / len(scores)
                result[game_name]["platforms"][platform_name] = {
                    "average_score": avg_score,
                    "count": platform_counts[platform_name]
                }

        # Calculate average score for the game
        all_scores = [outlet_data["score"] for outlet_data in data["outlets"].values() if isinstance(outlet_data["score"], (int, float))]
        if all_scores:
            result[game_name]["average_score"] = sum(all_scores) / len(all_scores)

    # Save to JSON
    with open(path, 'w') as json_file:
        json.dump(result, json_file, indent=4)
    print(f"Reviews data saved to {path}")

# Load platforms data from platforms.json
def load_platforms_data(filename="data/platforms.json"):
    with open(filename, 'r') as f:
        platforms_data = json.load(f)
    platforms_map = {platform['id']: platform['name'] for platform in platforms_data}
    return platforms_map

# Create Kafka consumer
consumer = KafkaConsumer(
    'game_reviews',  # Topic name
    bootstrap_servers='localhost:29092',  # Broker address
    auto_offset_reset='earliest',  # Start reading from the earliest offset
    value_deserializer=lambda v: json.loads(v.decode('utf-8'))  # Data deserialization
)

# Initialize necessary variables
outlets = {
    "IGN": 56,
    "PC Gamer": 162,
    "GamesRadar+": 91,
    "Metro GameCentral": 75,
    "Eurogamer": 114,
    "Easy Allies": 394,
    "Game Informer": 35,
    "Forbes": 290,
    "GameSpot": 32,
    "Game Rant": 60
}

platforms_map = load_platforms_data("data/platforms.json")
reviews_data = []

print("Collecting data...")

# Continuously collect data from Kafka topic
for message in consumer:
    game_review = message.value
    
    # Debug: Print the structure of the received message
    print(f"Received data structure: {game_review}")

    # Ensure the data format is correct
    if isinstance(game_review, dict) and 'game_name' in game_review and 'review' in game_review:
        game_name = game_review['game_name']
        review = game_review['review']
        reviews_data.append((game_name, review))
        
        # Print received data for debugging purposes
        print(f"Received data: {game_review}")

    # Process and save reviews after collecting a batch of reviews
    if len(reviews_data) >= 7:  # Adjust the batch size as needed
        process_and_save_reviews(reviews_data, outlets, platforms_map)
        reviews_data.clear()  # Clear the list to start collecting new data

consumer.close()


Received data structure: {'game_name': 'Game1', 'review': ['Game1', {'_id': '5d866e6893765d677873993f', 'magic': 0, 'isChosen': False, 'title': 'Dying Light Review: A Toothless Zombie Romp', 'publishedDate': '2018-10-17T00:00:00.000Z', 'externalUrl': 'https://gideonsgaming.com/2018/10/17/throw-back-review-dying-light/', 'snippet': "The free-running is fantastic, the combat feels excellent, and the crafting and skill system are well designed. Dying Light Plays well, and the story is entertaining, even if it is fairly basic. The gunplay definitely sours the experience when it comes up, and the lack of any real consequence for failure tarnishes every mechanic of the game. If that doesn't turn you off then there is a lot to love about this zombie romp", 'language': 'en-us', 'score': 78, 'npScore': 100, 'isQuoteManual': True, 'Authors': [{'_id': '64316a9934e32f8b008350bc', 'id': 5900, 'name': 'Joseph Pugh', 'image': True, 'imageSrc': {'og': 'critic/5900/o/542eqvCP.jpg', 'sm': 'critic/5900/g

## Web Scraping

In [ ]:
with open("data/game_ids.json", "r") as json_file:
    df_names = json.load(json_file)
    filtered_game_names = df_names["filtered_game_names"]

# List of game names to scrape (parameter: filtered_game_names) mocking data
# filtered_game_names = [
    # "grand theft AUTO v",
    # "Elden Ring",
    # "Wii Sports",
    # "Microsoft Flight Simulator",
    # "Cooking Mama",
    # "StardEW valley",
    # "Resident Evil 5",
    # "Unga Bunga doogaloo"
    # Add more game names here
# ]

# Base URL
base_url = "https://www.backloggd.com/games/"

# Initialize a list to store game names and their scores
data = []

# Loop through each game name in the list
for game_name in filtered_game_names:
    # Remove any colons from the game name
    game_name_clean = game_name.replace(':', '')
    
    # Create the URL by replacing spaces with hyphens and converting to lowercase
    url = f"{base_url}{game_name_clean.replace(' ', '-').lower()}/"
    
    # Send a GET request to the URL
    response = requests.get(url)
    
    # Check if the request was successful
    if response.status_code == 200:
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find the div with id 'score'
        score_div = soup.find('div', id='score')
        
        # If the score_div is found, extract the score
        if score_div:
            score = score_div.find('h1').text.strip()
            try:
                # Convert score to float and divide by 5 to get the percentage
                score_percentage = float(score) / 5 * 100
                # Append the game name and score to the data list
                data.append({'Game Name': game_name, 'Score': score_percentage})
            except ValueError:
                # Handle the case where the score is not a valid number
                print(f"Invalid score for game: {game_name}")
        else:
            print(f"Score not found for game: {game_name}")
    else:
        print(f"Failed to retrieve data for game: {game_name}")

# Create a DataFrame from the data list
df = pd.DataFrame(data)

# Save the DataFrame to a CSV file
df.to_csv('data/user_scores.csv', index=False)

print("Data saved to user_scores.csv")